**Ingeniería Civil Informática | Universidad Andrés Bello**

**Curso: Optimización**

**Equipo de Trabajo:**
* Eduardo Díaz
* Felipe Sanhueza
* Jean Guerrero

In [ ]:
!pip install -q amplpy
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(modules=["highs"], license_uuid="default")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.7 MB/s eta 0:00:00
Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


# Instrucciones:
Deben entregar un informe con su resolución y con pantallas del código en AMPL/Colab/Jypeter. Hacer un análisis de sensibilidad de los resultados y de las restricciones. Se debe enviar vía CANVAS y entregar informe impreso el día de la solemne. Deben resolver 6 de los ejercicios propuestos. El colab generado, debe ser subido en conjunto con el informe en .ZIP.

AE1. Modelar matemáticamente problemas clásicos de programación lineal entera mixta para establecer su complejidad. A


#1)

La Química fabrica tres productos químicos: A,B y C. Estos productos se obtienen por medio de 2 procesos de producción. El desarrollo del primer proceso durante 1 hora cuesta 40 USD y genera 3 unidades de A, una de B y y una de C. Efectuar el segundo proceso durante 1 hora cuesta 10 USD, y se obtienen una unidad de A y una de B. Para cumplir con las demandas de los clientes se tienen que producir todos los días por lo menos 40 unidades de A, 15  de B y 5 de C. Determinar un plan de producción diario, que minimice el costo de cumplir las demandas diarias de la Química.



# El planteamiento

Este modelo se basa en los procesos, y la cantidad de productos finales. A partir de ahí, la estrategia es estructurar restricciones que sumen el rendimiento por hora de cada proceso para garantizar que se cumpla la demanda mínima de los químicos A, B y C.

In [48]:
modelo_quimica = """
# Variables
var x1 >= 0;
var x2 >= 0;

# Función Objetivo
minimize Costo_Total: 40 * x1 + 10 * x2;

# Restricciones para cumplir la demanda de los químicos
s.t.
r1: 3 * x1 + 1 * x2 >= 40;
r2: 1 * x1 + 1 * x2 >= 15;
r3: 1 * x1 >= 5;
"""

In [49]:
ampl.reset()
ampl.eval(modelo_quimica)
ampl.option["solver"] = "highs"
ampl.solve()

print("Costo minimizado: ", ampl.get_objective('Costo_Total').value())
print("Proceso 1: ", ampl.get_variable('x1').value())
print("Proceso 2: ", ampl.get_variable('x2').value())

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations
Costo minimizado:  450.0
Proceso 1:  5.0
Proceso 2:  25.0


# 2)

Todas las semanas, Charcha puede comprar cantidades ilimitadas de materia prima a 6 dólares la libra. Cada libra de materia prima comprada se puede usar para elaborar el insumo 1 o el 2. Cada libra de materia prima rinde 2 oz del insumo 1, requiere 2 h. de tiempo de proceso e incurre en costos de proceso por 2 dólares. Cada libra de materia prima rinde 3 oz del insumo 2, requiere 2 h. de tiempo de proceso y sus costos de proceso son de 4 dólares. Se dispone de dos procesos de producción. El proceso 1 requiere 2 h., 2 oz de insumo 1 y 1 oz del insumo 2. Cuesta 1 dólar ejecutar el proceso 1. Cada vez que el proceso 1 se efectúa, se produce 1 oz del producto A y 1 oz de desecho líquido. Cada vez que el proceso 2 se efectúa, se requieren 3 h. de proceso, 2 oz del insumo 2 y 1 oz del insumo 1. El proceso 2 rinde 1 oz del producto B y 0.8 oz de desecho líquido. Los costos del proceso 2 son 8 dólares. Charcha puede tirar sus desechos líquidos en el río Port Charles, o bien, usar el desecho para elaborar el producto C o el producto D. De acuerdo con los reglamentos gubernamentales, Charcha tiene permitido derramar al río cuando mucho 1 000 oz a la semana. El costo por producir una onza del producto C es de 4 dólares, y se vende en 11 dólares. Se requiere una hora de tiempo de proceso, 2 oz del insumo 1 y 0.8 oz de desecho líquido para producir una onza del producto C. Cuesta 5 dólares fabricar una unidad del producto D y se vende en 7 dólares. Una hora de tiempo de proceso, 2 oz del insumo 2 y 1.2 oz de desecho líquido es lo que se requiere para fabricar una onza del producto D. Todas las semanas se venden, cuando mucho, 5 000 oz del producto A y 5 000 oz del producto B, pero la demanda semanal de los productos C y D es ilimitada. El producto A se vende a 18 dólares la onza y cada onza del producto B se vende en 24 dólares. Se dispone cada semana de 6 000 h. de tiempo de proceso. Formule un PL cuya solución le señale a Charcha cómo maximizar las utilidades semanales.

# Planteamiento

La idea es definir todos las variables: la materia prima debe cubrir la necesidad de insumos, los insumos deben sostener la fabricación de productos, y los desechos líquidos se reciclan o se vierten al río.

In [50]:
modelo_charcha = """
# Variables de Decisión
var RM1 >= 0;              # Libras de materia prima usadas para crear Insumo 1
var RM2 >= 0;              # Libras de materia prima usadas para crear Insumo 2
var P1 >= 0, <= 5000;      # Onzas producidas de A (Proceso 1 tiene tope de 5000)
var P2 >= 0, <= 5000;      # Onzas producidas de B (Proceso 2 tiene tope de 5000)
var C >= 0;                # Onzas producidas de C
var D >= 0;                # Onzas producidas de D
var Palrio >= 0, <= 1000;  # Desecho tirado al río (Máximo 1000 oz)

# Funcion objetivo
maximize Utilidad:
    (18*P1 + 24*P2 + 11*C + 7*D) - (8*RM1 + 10*RM2 + 1*P1 + 8*P2 + 4*C + 5*D);

# Restricciones
s.t.
r1: 2*RM1 + 2*RM2 + 2*P1 + 3*P2 + 1*C + 1*D <= 6000;
r2: 2*RM1 >= 2*P1 + 1*P2 + 2*C;
r3: 3*RM2 >= 1*P1 + 2*P2 + 2*D;
r4: 1*P1 + 0.8*P2 == Palrio + 0.8*C + 1.2*D;
"""

In [51]:
ampl.reset()
ampl.eval(modelo_charcha)
ampl.option["solver"] = "highs"
ampl.solve()

print ("Utilidad maximizada: ", round(ampl.get_objective('Utilidad').value(),2), "lbs")
print ("Materia Prima Insumo 1:   ", round(ampl.get_variable('RM1').value(),2), "lbs")
print ("Materia Prima Insumo 2: ", round(ampl.get_variable('RM2').value(),2))
print ("Onzas de Producto A: ", round(ampl.get_variable('P1').value(),2))
print ("Onzas de Producto B: ", round(ampl.get_variable('P2').value(),2))
print ("Onzas de Producto C: ", round(ampl.get_variable('C').value(),2))
print ("Onzas de Producto D: ", round(ampl.get_variable('D').value(),2))
print ("Desecho tirado al río: ", round(ampl.get_variable('Palrio').value(),2), "oz")

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 6366.336634
6 simplex iterations
0 barrier iterations
Utilidad maximizada:  6366.34 lbs
Materia Prima Insumo 1:    1356.44 lbs
Materia Prima Insumo 2:  386.14
Onzas de Producto A:  1158.42
Onzas de Producto B:  0.0
Onzas de Producto C:  198.02
Onzas de Producto D:  0.0
Desecho tirado al río:  1000.0 oz


# 3)

Usted ha sido designado administrador de la refinería de Melrose. Esta refinería produce gasolina y aceite combustible a partir del petróleo crudo. La gasolina se vende a 8 dólares el barril, y debe tener un 'grado' promedio de por lo menos 9. El aceite combustible se vende a 6 dólares el barril y debe tener un 'grado' de por lo menos 7. Se pueden vender, cuando mucho, 2 000 barriles de gasolina y 600 barriles de aceite combustible. El crudo que está por llegar puede ser procesado por medio de uno de tres métodos distintos. El rendimiento por barril y el costo por barril de cada método de proceso se proporcionan en la tabla 32. Por ejemplo, si se refina un barril del crudo que llega por el método 1, cuesta 3.40 dólares y da un rendimiento de 0.2 de barril de grado 6, 0.2 de barril de grado 8 y 0.6 de barril de grado 10. Antes de ser procesado para obtener gasolina y aceite combustible, los grados 6 y 8 podrían enviarse al desintegrador catalítico para mejorar su calidad. Por 1.30 dólares el barril, un barril de grado 6 puede ser fraccionado en 1 barril de grado 8. Por 2 dólares el barril, un barril de grado 8 se fracciona en un barril de grado 10. Cualquier crudo excedente, procesado o fraccionado, que ya no se pueda utilizar para aceite combustible o gasolina, se debe desechar a un costo de 0.20 de dólar por barril. Determine cómo maximizar la utilidad de la refinería."

Método	Grado 6	Grado 8	Grado 10	Costo USD
1	0.2	0.2	0.6	3.4
2	0.3	0.3	0.4	3
3	0.4	0.4	0.2	2.6




# Planteamiento

se va a calcular un promedio con divisiones, se multiplica el volumen total esperado por el grado mínimo exigido. Además, se formulan los barriles que se procesan, desechan o mejoraran.

In [42]:
modelo_refineria = """
# Variables de Decisión
var M1 >= 0;  # Barriles de crudo procesados por Método 1
var M2 >= 0;  # Barriles de crudo procesados por Método 2
var M3 >= 0;  # Barriles de crudo procesados por Método 3
var C68 >= 0;   # Barriles de Grado 6 mejorados a Grado 8
var C810 >= 0;  # Barriles de Grado 8 mejorados a Grado 10
var Gas6 >= 0;   # Grado 6 usado para Gasolina
var Gas8 >= 0;   # Grado 8 usado para Gasolina
var Gas10 >= 0;  # Grado 10 usado para Gasolina
var FO6 >= 0;    # Grado 6 usado para Aceite Combustible
var FO8 >= 0;    # Grado 8 usado para Aceite Combustible
var FO10 >= 0;   # Grado 10 usado para Aceite Combustible
var D6 >= 0;   # Grado 6 desechado
var D8 >= 0;   # Grado 8 desechado
var D10 >= 0;  # Grado 10 desechado
var Gas >= 0, <= 2000;  # Total Gasolina a vender (Máximo 2000)
var FO >= 0, <= 600;    # Total Aceite Combustible a vender (Máximo 600)

# 2. Función Objetivo: Ingresos (Ventas) - Costos (Procesamiento + Mejora + Desecho)
maximize Utilidad:
    (8*Gas + 6*FO) - (3.4*M1 + 3.0*M2 + 2.6*M3) - (1.3*C68 + 2.0*C810) - 0.2*(D6 + D8 + D10);

s.t.
# 3. Restricciones de Definición (El total de la mezcla es la suma de sus partes)
r1: Gas == Gas6 + Gas8 + Gas10;
r2:  FO  == FO6  + FO8  + FO10;

# 4. Restricciones de Balance de Masa (Lo que se Produce + Lo que se Mejora hacia acá == Lo que se usa + Lo que se Mejora desde acá + Desecho)
r3:  0.2*M1 + 0.3*M2 + 0.4*M3          == C68  + Gas6  + FO6  + D6;
r4:  0.2*M1 + 0.3*M2 + 0.4*M3 + C68    == C810 + Gas8  + FO8  + D8;
r5: 0.6*M1 + 0.4*M2 + 0.2*M3 + C810   ==        Gas10 + FO10 + D10;

# 5. Restricciones de Calidad Promedio (El truco de no usar divisiones)
r6: 6*Gas6 + 8*Gas8 + 10*Gas10 >= 9*Gas;
r7:  6*FO6  + 8*FO8  + 10*FO10  >= 7*FO;
"""

In [43]:
ampl.reset()
ampl.eval(modelo_refineria)
ampl.option["solver"] = "highs"
ampl.solve()

print ("Utilidad Maximizada: $", round(ampl.get_objective('Utilidad').value(), 2))
print ("Gasolina Vendida: ", round(ampl.get_variable('Gas').value(),2), "barriles")
print ("Aceite Combustible Vendido: ", round(ampl.get_variable('FO').value(),2), "barriles")
print ("Método 1: ", round(ampl.get_variable('M1').value(),2), "barriles")
print ("Método 2: ", round(ampl.get_variable('M2').value(),2), "barriles")
print ("Método 3: ", round(ampl.get_variable('M3').value(),2), "barriles")
print ("Mejora Grado 6 a 8: ", round(ampl.get_variable('C68').value(),2), "barriles")
print ("Mejora Grado 8 a 10: ", round(ampl.get_variable('C810').value(),2), "barriles")
print (f"Total Desechado: {ampl.get_variable('D6').value() + ampl.get_variable('D8').value() + ampl.get_variable('D10').value()} barriles")

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 11230
8 simplex iterations
0 barrier iterations
Utilidad Maximizada: $ 11230.0
Gasolina Vendida:  2000.0 barriles
Aceite Combustible Vendido:  600.0 barriles
Método 1:  0.0 barriles
Método 2:  2400.0 barriles
Método 3:  200.0 barriles
Mejora Grado 6 a 8:  500.0 barriles
Mejora Grado 8 a 10:  0.0 barriles
Total Desechado: 0.0 barriles


# 4)

Donovan Enterprises fabrica licuadoras. Durante los cuatro trimestres siguientes se tiene que cumplir (a tiempo) con la siguiente demanda de licuadoras: trimestre 1, 4 000; trimestre 2, 2 000; trimestre 3, 3 000; trimestre 4, 10 000. Cada empleado de Donovan trabaja tres trimestres del año y tiene un trimestre libre. Por consiguiente, un empleado podría trabajar durante los trimestres 1, 2 y 4, y tener libre el trimestre 3. Cada empleado recibe un salario de 30 000 dólares al año, y (si trabaja) produce hasta 500 licuadoras en un trimestre. Al final de cada trimestre, Donovan incurre en un costo por guardar los artículos de 30 dólares por unidad sobre cada licuadora en inventario. Plantee un PL para ayudar a Donovan a minimizar los costos (mano de obra e inventario) en el cumplimiento (a tiempo) de la demanda del año próximo. Hay en existencia 600 licuadoras a principios del trimestre 1."

# Planteamiento

La estrategia utiliza la siguiente lógica: primero, definir a los trabajadores según su trimestre de descanso para calcular por descarte quiénes producen en cada periodo y segundo, usar una ecuación que conecta los trimestres: inventario anterior + producción actual = demanda + inventario nuevo.

In [52]:
modelo_licuadoras = """
set TRIMESTRES := 1..4;

param demanda{TRIMESTRES};
param inv_inicial := 600;
param capacidad_empleado := 500;
param costo_sueldo := 30000;
param costo_inventario := 30;

# Tipo de empleado según su trimestre de descanso
var W{TRIMESTRES} >= 0;

# Producción en cada trimestre
var P{TRIMESTRES} >= 0;

# Inventario al final de cada trimestre (incluimos el mes 0 para el inicio)
var I{0..4} >= 0;

minimize Costo_Total:
    sum{i in TRIMESTRES} costo_sueldo * W[i] +
    sum{t in TRIMESTRES} costo_inventario * I[t];

# 4. Restricciones
s.t.
# Condición del inventario inicial (Trimestre 0)
inicio_inventario:
    I[0] == inv_inicial;

# Capacidad Máxima: La producción no puede superar lo que arman los activos
limite_produccion{t in TRIMESTRES}:
    P[t] <= capacidad_empleado * sum{i in TRIMESTRES: i != t} W[i];

# Conservación de Flujo: Inventario anterior + Producción == Demanda + Inventario nuevo
balance{t in TRIMESTRES}:
    I[t-1] + P[t] == demanda[t] + I[t];

data;
param demanda :=
    1 4000
    2 2000
    3 3000
    4 10000;
"""

In [53]:
ampl.reset()
ampl.eval(modelo_licuadoras)
ampl.option["solver"] = "highs"
ampl.solve()

print("Costo Total Minimizado: ", ampl.get_objective('Costo_Total').value())
for t in [1, 2, 3, 4]:
    print("--- Trimestre", t, "---")
    print("Producción: ", ampl.get_variable('P')[t].value())
    print("Inventario: ", ampl.get_variable('I')[t].value())

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 495000
6 simplex iterations
0 barrier iterations
Costo Total Minimizado:  495000.0
--- Trimestre 1 ---
Producción:  3400.0
Inventario:  0.0
--- Trimestre 2 ---
Producción:  2000.0
Inventario:  0.0
--- Trimestre 3 ---
Producción:  6500.0
Inventario:  3500.0
--- Trimestre 4 ---
Producción:  6500.0
Inventario:  0.0


# 5)

Un grupo de investigación de mercado necesita detectar por lo menos a 150 esposas, 120 esposos, 100 varones adultos solteros y 110 mujeres adultas solteras mediante una encuesta telefónica. Cuesta 2 dólares hacer una llamada en el día y (debido a los costos de mano de obra más altos) 5 dólares una llamada por la noche. Los resultados se dan en la tabla. Debido a que el personal es limitado, cuando mucho la mitad de todas las llamadas pueden ser nocturnas. Plantee un PL para minimizar el costo de completar la encuesta."


Persona que contesta	% de llamadas en el día	% de llamadas en el noche
Esposa	30	30
Esposo	10	30
Varón Soltero	10	15
Mujer Soltera	10	20
Nadie	40	5


# Planteamiento

 Las variables de decisión son las llamadas de día y noche, al mismo tiempo ignorando las llamadas de Nadie, ya que no existe una exigencia sobre ellas y su costo ya está calculado al intentar la llamada.

In [54]:
modelo_encuesta = """
var X1 >= 0;  # dia
var X2 >= 0;  # noche

# Función objetivo
minimize Costo_Total:
    2*X1 + 5*X2;

# Restricciones
s.t.
r1: 0.3*X1 + 0.3*X2 >= 150;
r2: 0.1*X1 + 0.3*X2 >= 120;
r3: 0.1*X1 + 0.15*X2 >= 100;
r4: 0.1*X1 + 0.2*X2 >= 110;
r5: X2 <= 0.5*(X1 + X2);

"""

In [55]:
ampl.reset()
ampl.eval(modelo_encuesta)
ampl.option["solver"] = "highs"
ampl.solve()

print("Costo total: ", round(ampl.get_objective('Costo_Total').value(),2))
print("Llamadas de Día:   ", round(ampl.get_variable('X1').value(),2))
print("Llamadas de Noche: ", round(ampl.get_variable('X2').value(),2))

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2300
4 simplex iterations
0 barrier iterations
Costo total:  2300.0
Llamadas de Día:    900.0
Llamadas de Noche:  100.0


# 6)

Brady Corporation fabrica alacenas. Requiere cada semana 90 000 pies cúbicos de tablones. La compañía puede conseguir madera de dos maneras: primero, la podría comprar con un proveedor y secarla en el horno del proveedor. Segundo, podría cortar troncos en sus propios terrenos, cortarlos en tablones en su aserradero y, por último, secarlos en su propio horno. Compra y Calidad de Tablones: Brady puede comprar tablones grado 1 o grado 2. Los tablones grado 1 cuestan 3 dólares por pie cúbico, y cuando se secan rinden 0.7 pies cúbicos de madera útil. Los tablones grado 2 cuestan 7 dólares el pie cúbico, y luego de secarlos rinden 0.9 pies cúbicos de madera útil.
Procesamiento de Troncos Propios: A la compañía le cuesta 3 dólares cortar los troncos. Después de cortar y secar un tronco, éste rinde 0.8 pies cúbicos de tablones. Costos de Operación: Brady gasta 4 dólares por pie cúbico de tablones secados. Además, cuesta 2.50 dólares por pie cúbico de troncos enviados al aserradero. Capacidades y Restricciones: El aserradero puede procesar cada semana hasta 35 000 pies cúbicos de tablones. Se pueden comprar cada semana hasta 40 000 pies cúbicos de tablones grado 1 y hasta 60 000 pies cúbicos del grado 2. Se dispone cada semana de 40 h para secar los tablones. Tiempos de Secado: El tiempo que se requiere para secar 1 pie cúbico de madera es el siguiente: Grado 1: 1.2 segundos (s).Grado 2: 0.8 s. Troncos: 1.3 s. Determine un PL (Programación Lineal) que ayude a Brady a minimizar el costo a la semana por cumplir con la demanda de tablones procesados.



# Planteamiento

 Primero revisamos bien los parametros del ejercicio para sacar las variables: todos los costos en cadena (compra, corte, aserradero y secado) en un costo total para la función objetivo, y al mismo tiempo buscamos las restricciones, transformando el límite del horno de horas a segundos para comparar el rendimiento.

In [56]:
modelo_brady = """
# variables de decisión
var tablaG1  >= 0;
var tablaG2  >= 0;
var troncoPropio >= 0;

# Función objetivo
minimize Costo_Total:
(3+4) * tablaG1 + (7+4) * tablaG2 + (3 + 2.5 + 4) * troncoPropio;

# Restricciones
s.t.
r1: 0.7 * tablaG1 + 0.9 * tablaG2 + 0.8 * troncoPropio >= 90000;
r2: 1.2 * tablaG1 + 0.8 * tablaG2 + 1.3 * troncoPropio <= 144000 ;
r3: tablaG1 <= 40000;
r4: tablaG2 <= 60000;
r5: troncoPropio <= 35000;
"""

In [57]:
ampl.reset()
ampl.eval(modelo_brady)
ampl.option["solver"] = "highs"
ampl.solve()

print ("Costo total: ", round(ampl.get_objective('Costo_Total').value(),2))
print ("tablas grado 1:   ", round(ampl.get_variable('tablaG1').value(),2))
print ("tablas grado 2: ", round(ampl.get_variable('tablaG2').value(),2))
print ("troncos propios: ", round(ampl.get_variable('troncoPropio').value(),2))

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 1028055.556
0 simplex iterations
0 barrier iterations
Costo total:  1028055.56
tablas grado 1:    40000.0
tablas grado 2:  37777.78
troncos propios:  35000.0


# 7)

Un centro de reciclaje industrial utiliza dos chatarras de aluminio, A y B, para producir una aleación especial. La chatarra A contiene 6% de aluminio, 3% de silicio, y 4% de carbón. La chatarra B contiene 3% de aluminio, 6% de silicio, y 3% de carbón. Los costos por tonelada de las chatarras A y B son de $100 y $80, respectivamente. Las especificaciones de la aleación especial requieren que (1) el contenido de aluminio debe ser mínimo de 3% y máximo de 6%; (2) el contenido de silicio debe ser de entre 3 y 5%, y (3) el contenido de carbón debe ser de entre 3 y 7%. Determine la mezcla óptima de las chatarras que deben usarse para producir 1000 toneladas de la aleación.

# Planteamiento

 Se busca a traves de las toneladas de chatarra A y B, minimizar los costos de compra, cumpliendo con la restricción de exactamente 1000 toneladas.Además, se establecen cotas inferiores y superiores para cada elemento aluminio, silicio y carbón.

In [60]:
modelo_chatarra = """
var A >= 0; # Toneladas chatarra A
var B >= 0; # Toneladas chatarra B

# Función objetivo
minimize Costo_Total:
100*A + 80*B;

# Restricciones
r1: A + B = 1000;
r2: 0.06*A + 0.03*B >= 0.03 * 1000; # Aluminio Minimo
r3: 0.06*A + 0.03*B <= 0.06 * 1000; # Aluminio Maximo
r4: 0.03*A + 0.06*B >= 0.03 * 1000; # Silicio Minimo
r5: 0.03*A + 0.06*B <= 0.05 * 1000; # Silicio Maximo
r6: 0.04*A + 0.03*B >= 0.03 * 1000; # Carbon Minimo
r7: 0.04*A + 0.03*B <= 0.07 * 1000; # Carbon Maximo
"""

In [61]:
ampl.reset()
ampl.eval(modelo_chatarra)
ampl.option["solver"] = "highs"
ampl.solve()

print ("Costo total: ", round(ampl.get_objective('Costo_Total').value(),2))
print("Chatarra A:   ", round(ampl.get_variable('A').value(),2))
print("Chatarra B: ", round(ampl.get_variable('B').value(),2))

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 86666.66667
0 simplex iterations
0 barrier iterations
Costo total:  86666.67
Chatarra A:    333.33
Chatarra B:  666.67
